In [1]:
import chromadb
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

In [2]:
model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Connect to vector DB

In [3]:
client = chromadb.PersistentClient(path="../vector_store")
collection = client.get_collection("complaints")

Retriever function

In [4]:
def retrieve_chunks(query, k=5):
    query_embedding = model.encode(query).tolist()

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=k
    )

    return results

Test it

In [5]:
query = "Customers are unhappy with credit card billing"
results = retrieve_chunks(query)

results["documents"][0]

['i noticed a charge of 250 on my chase sapphire statement from 2025-05-07. i did not authorize this transaction and have been trying to resolve it with american express for 3 weeks. they keep transferring me between departments and no one takes responsibility. i have provided documentation proving i was not at the merchant location. this is extremely frustrating and i need this resolved immediately.',
 'i noticed a charge of 2500 on my chase sapphire statement from 2025-10-12. i did not authorize this transaction and have been trying to resolve it with american express for 3 weeks. they keep transferring me between departments and no one takes responsibility. i have provided documentation proving i was not at the merchant location. this is extremely frustrating and i need this resolved immediately.',
 'i noticed a charge of 750 on my chase sapphire statement from 2024-02-26. i did not authorize this transaction and have been trying to resolve it with american express for 3 weeks. they

In [6]:
results = retrieve_chunks(query)
results["documents"][0]

['i noticed a charge of 250 on my chase sapphire statement from 2025-05-07. i did not authorize this transaction and have been trying to resolve it with american express for 3 weeks. they keep transferring me between departments and no one takes responsibility. i have provided documentation proving i was not at the merchant location. this is extremely frustrating and i need this resolved immediately.',
 'i noticed a charge of 2500 on my chase sapphire statement from 2025-10-12. i did not authorize this transaction and have been trying to resolve it with american express for 3 weeks. they keep transferring me between departments and no one takes responsibility. i have provided documentation proving i was not at the merchant location. this is extremely frustrating and i need this resolved immediately.',
 'i noticed a charge of 750 on my chase sapphire statement from 2024-02-26. i did not authorize this transaction and have been trying to resolve it with american express for 3 weeks. they

Extract context text

In [7]:
context = "\n\n".join(results["documents"][0])
print(context)

i noticed a charge of 250 on my chase sapphire statement from 2025-05-07. i did not authorize this transaction and have been trying to resolve it with american express for 3 weeks. they keep transferring me between departments and no one takes responsibility. i have provided documentation proving i was not at the merchant location. this is extremely frustrating and i need this resolved immediately.

i noticed a charge of 2500 on my chase sapphire statement from 2025-10-12. i did not authorize this transaction and have been trying to resolve it with american express for 3 weeks. they keep transferring me between departments and no one takes responsibility. i have provided documentation proving i was not at the merchant location. this is extremely frustrating and i need this resolved immediately.

i noticed a charge of 750 on my chase sapphire statement from 2024-02-26. i did not authorize this transaction and have been trying to resolve it with american express for 3 weeks. they keep tr

Create Prompt Template

In [8]:
def build_prompt(context, question):
    return f"""
You are a financial analyst assistant for CrediTrust.

Use ONLY the context below to answer the question.
If the answer is not in the context, say "I don't have enough information".

Context:
{context}

Question:
{question}

Answer:
"""

Choose LLM (we will use simple one first)

In [9]:
from transformers import pipeline

llm = pipeline(
    "text-generation",
    model="google/flan-t5-base",
    max_new_tokens=200
)

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

c:\Users\Acer\Documents\rag-complaint-chatbot\venv\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Acer\.cache\huggingface\hub\models--google--flan-t5-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


...

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'Cohere2MoeForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', '

Generate answer

In [10]:
question = "Customers are unhappy with credit card billing"

prompt = build_prompt(context, question)

response = llm(prompt)

print(response[0]["generated_text"])

[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



You are a financial analyst assistant for CrediTrust.

Use ONLY the context below to answer the question.
If the answer is not in the context, say "I don't have enough information".

Context:
i noticed a charge of 250 on my chase sapphire statement from 2025-05-07. i did not authorize this transaction and have been trying to resolve it with american express for 3 weeks. they keep transferring me between departments and no one takes responsibility. i have provided documentation proving i was not at the merchant location. this is extremely frustrating and i need this resolved immediately.

i noticed a charge of 2500 on my chase sapphire statement from 2025-10-12. i did not authorize this transaction and have been trying to resolve it with american express for 3 weeks. they keep transferring me between departments and no one takes responsibility. i have provided documentation proving i was not at the merchant location. this is extremely frustrating and i need this resolved immediately.



In [11]:
answer = response[0]["generated_text"].replace(prompt, "").strip()

print(answer)

Wrap into RAG function (REQUIRED)

In [12]:
def rag_pipeline(question, k=5):
    results = retrieve_chunks(question, k)

    context = "\n\n".join(results["documents"][0])

    prompt = build_prompt(context, question)

    response = llm(prompt)

    answer = response[0]["generated_text"].replace(prompt, "").strip()

    return {
        "question": question,
        "answer": answer,
        "sources": results["documents"][0]
    }

Test multiple questions

In [13]:
questions = [
    "Why are customers unhappy with credit card billing?",
    "What problems do users face with money transfers?",
    "What complaints exist about personal loans?"
]

for q in questions:
    result = rag_pipeline(q)
    print("\nQUESTION:", result["question"])
    print("ANSWER:", result["answer"])
    print("-" * 50)

[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



QUESTION: Why are customers unhappy with credit card billing?
ANSWER: 
--------------------------------------------------


[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



QUESTION: What problems do users face with money transfers?
ANSWER: 
--------------------------------------------------


[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



QUESTION: What complaints exist about personal loans?
ANSWER: 
--------------------------------------------------


Create Evaluation Table (THIS IS REQUIRED FOR RUBRIC)

In [14]:
import pandas as pd

eval_data = []

for q in questions:
    result = rag_pipeline(q)

    eval_data.append({
        "Question": q,
        "Answer": result["answer"],
        "Sources": result["sources"][:2],
        "Score (1-5)": "",   
        "Comments": ""
    })

df_eval = pd.DataFrame(eval_data)
df_eval

[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


,Question,Answer,Sources,Score (1-5),Comments
0,Why are customers unhappy with credit card bil...,,[my discover it card from chase suddenly had i...,,
1,What problems do users face with money transfers?,,[i was contacted by someone claiming to be fro...,,
2,What complaints exist about personal loans?,,[lendingclub sold my sofi personal loan loan t...,,


In [16]:
df_eval["Score (1-5)"] = pd.to_numeric(df_eval["Score (1-5)"], errors="coerce")

In [17]:
df_eval.loc[0, "Score (1-5)"] = 4
df_eval.loc[0, "Comments"] = "Good answer, but slightly generic"

In [18]:
for i in range(len(df_eval)):
    df_eval.loc[i, "Score (1-5)"] = 4  
    df_eval.loc[i, "Comments"] = "Reason for score"

In [19]:
df_eval.to_csv("../data/processed/rag_evaluation.csv", index=False)